# XFeat — exact Thermometer acceleration (2026-10-05)

**런타임 → 모두 실행. 맨 앞에서 POPCOUNT 사용 여부가 출력됩니다.**
`YES_NATIVE_POPCOUNT`는 실제 생성된 기계어에서 POPCNT/VPOPCNT를 찾았다는 뜻입니다.
`NOT_CONFIRMED`는 미확인이며, LLVM ctpop 존재만으로 하드웨어 명령 사용을 단정하지 않습니다.

이번 기본 실험: 원본 cosine / 기존 dense Thermometer / 양방향 동시 갱신 / 블록 32·64·128 / 병렬 블록 64 (1·2 threads).
- **양방향 동시 갱신:** pair 거리를 한 번 계산하고 A측·B측 최솟값을 바로 갱신. 전체 거리 행렬 저장 제거.
- **블록 계산:** descriptor 배열의 작은 타일을 순서대로 처리해 캐시 재사용을 시도. 공간 BLOB이나 후보 제거가 아니며 모든 pair를 비교.
- 병렬 버전은 작업별 B측 최솟값을 저장 후 병합하여 race 방지. 동점은 최소 원래 KP index.
- 정확도는 기존 Thermometer와 완전히 같아야 하며, BOTH NN와 MNN를 모든 평가 쌍에서 assert합니다.
- 마지막 **12~13번**에서 새 결과표와 `under_5ms`를 확인하세요. 5 ms 달성을 보장하지 않습니다.
- 기존 Grouping·Blob 코드 보존. `RUN_LEGACY_EXPERIMENTS=False`, `RUN_BLOB_BENCHMARKS=False`로 기본 실행 생략.
- 전처리/cached matching/전체 시간 분리. JIT·XFeat 추출 제외. 이전 출력은 제거하였으며 이전 저장 결과 폴더는 보존됩니다.


## 0. POPCOUNT 기계어 진단 — 데이터 로딩 전에 실행

In [24]:
%pip -q install numba numpy
import numpy as np
from numba import njit,prange,set_num_threads
set_num_threads(min(2,__import__("numba").config.NUMBA_NUM_THREADS))

@njit(parallel=True, cache=False)
def pairwise_uint8_abs(code_1, code_2):
    out = np.empty((code_1.shape[0], code_2.shape[0]), np.uint16)
    for i in prange(code_1.shape[0]):
        for j in range(code_2.shape[0]):
            total = 0
            for k in range(code_1.shape[1]):
                a, b = int(code_1[i, k]), int(code_2[j, k])
                total += a - b if a >= b else b - a
            out[i, j] = total
    return out

def mutual_from_scores(scores, higher_is_better):
    if not scores.shape[0] or not scores.shape[1]:
        return np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int64)
    if higher_is_better:
        nearest_12, nearest_21 = scores.argmax(axis=1), scores.argmax(axis=0)
    else:
        nearest_12, nearest_21 = scores.argmin(axis=1), scores.argmin(axis=0)
    indices = np.arange(scores.shape[0])
    keep = nearest_21[nearest_12] == indices
    return indices[keep], nearest_12[keep]

# Packed thermometer: same 8-bin L1 distance, different computation.
from numba import types
from numba.extending import intrinsic
from llvmlite import ir

@intrinsic
def thermometer_popcount64(typing_context, value_type):
    if value_type != types.uint64:
        raise TypeError('thermometer_popcount64 expects uint64')
    signature = types.uint64(types.uint64)
    def codegen(context, builder, signature, arguments):
        function = builder.module.declare_intrinsic('llvm.ctpop', [ir.IntType(64)])
        return builder.call(function, arguments)
    return signature, codegen

@njit(cache=False)
def _pack_thermometer64(codes):
    packed = np.zeros((codes.shape[0], 7), dtype=np.uint64)
    for row in range(codes.shape[0]):
        for threshold in range(1, 8):
            word = np.uint64(0)
            for dimension in range(64):
                if codes[row, dimension] >= threshold:
                    word |= np.uint64(1) << np.uint64(dimension)
            packed[row, threshold - 1] = word
    return packed

def pack_thermometer64(codes):
    codes = np.asarray(codes)
    if codes.ndim != 2 or codes.shape[1] != 64:
        raise ValueError('Expected (N, 64) bin indices')
    if not np.issubdtype(codes.dtype, np.integer):
        raise ValueError('Bin indices must be integers')
    if codes.size and (codes.min() < 0 or codes.max() > 7):
        raise ValueError('Expected bin indices 0..7')
    return _pack_thermometer64(np.ascontiguousarray(codes, dtype=np.uint8))

@njit(parallel=True, cache=False)
def pairwise_thermometer64(packed_1, packed_2):
    distances = np.empty((packed_1.shape[0], packed_2.shape[0]), dtype=np.uint16)
    for i in prange(packed_1.shape[0]):
        for j in range(packed_2.shape[0]):
            total = np.uint64(0)
            for plane in range(7):
                total += thermometer_popcount64(packed_1[i, plane] ^ packed_2[j, plane])
            distances[i, j] = total
    return distances


import re,platform,json
import numba,llvmlite
from llvmlite import binding as llvm_binding

POPCOUNT_ASSEMBLY={}
def popcount_report(name,fn):
    asm='\n'.join(fn.inspect_asm(sig) for sig in fn.signatures)
    llvm='\n'.join(fn.inspect_llvm(sig) for sig in fn.signatures)
    # Match instruction at the start of an assembly line, never a symbol name.
    native=[line.strip() for line in asm.splitlines()
            if re.match(r'^\s*(?:popcnt[bwlq]?|vpopcnt[bwdq])\s',line)]
    arm=[line.strip() for line in asm.splitlines() if re.match(r'^\s*cnt\s+v\d+\.',line)]
    status='YES_NATIVE_POPCOUNT' if native else ('YES_ARM_VECTOR_CNT' if arm else 'NOT_CONFIRMED')
    row=dict(kernel=name,status=status,llvm_ctpop=('llvm.ctpop' in llvm),
             native_instruction_lines=len(native)+len(arm),scalar_popcnt_lines=sum(x.startswith('popcnt') for x in native),
             vector_popcnt_lines=sum(x.startswith('vpopcnt') for x in native),examples=(native+arm)[:4])
    POPCOUNT_ASSEMBLY[name]=asm
    print(name,':',status,'| LLVM ctpop:',row['llvm_ctpop'])
    for line in row['examples']:print('   ',line)
    if status=='NOT_CONFIRMED':print('   Native popcount not found by this diagnostic; inspect saved assembly. LLVM ctpop alone is not proof.')
    return row

print('CPU:',llvm_binding.get_host_cpu_name(),'architecture:',platform.machine())
print('Numba:',numba.__version__,'llvmlite:',llvmlite.__version__)
_features=llvm_binding.get_host_cpu_features()
print('CPU popcnt capability:',_features.get('popcnt','not reported'),
      '| JIT CPU override:',numba.config.CPU_NAME)
_probe_a=np.ascontiguousarray(np.array([[0,1,3,7,15,31,2**63],[2**64-1]*7],dtype=np.uint64))
_probe_b=np.zeros((1,7),np.uint64)
_probe_d=pairwise_thermometer64(_probe_a,_probe_b)
assert int(_probe_d[0,0])==16 and int(_probe_d[1,0])==448
print('PASS: popcount numerical check (including bit63 and 448).')
POPCOUNT_REPORTS=[popcount_report('existing_pairwise_thermometer64',pairwise_thermometer64)]


CPU: broadwell architecture: x86_64
Numba: 0.61.2 llvmlite: 0.44.0
CPU popcnt capability: True | JIT CPU override: None
PASS: popcount numerical check (including bit63 and 448).
existing_pairwise_thermometer64 : YES_NATIVE_POPCOUNT | LLVM ctpop: True
    popcntq	%r13, %rbx
    popcntq	%rax, %rax
    popcntq	%rbx, %rbx
    popcntq	%rax, %rax


## 1. 데이터와 설정
기존 seed=7, TOP_K=800, COCO 24장과 scene split을 유지합니다. Upload 모드에서는 서로 다른 장면들을 사용하세요. 학습 데이터와 검증 데이터에 같은 장면을 넣지 마세요.

In [25]:
%pip -q install opencv-python-headless numba pandas matplotlib threadpoolctl tqdm

RUN_LEGACY_EXPERIMENTS = False
RUN_BLOB_BENCHMARKS = False

import time
import io
import zipfile
import subprocess
import shutil
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from numba import njit, prange, set_num_threads, get_num_threads
from google.colab import files, drive

SEED = 7
TOP_K = 800
LEVELS = 8  # seven thresholds per descriptor dimension
CALIBRATION_WARPS_PER_IMAGE = 6
VALIDATION_WARPS_PER_IMAGE = 3
MAX_POSITIVES_PER_PAIR = 500
GEOMETRIC_THRESHOLD = 3.0
NUMBA_THREADS = 2
RUN_HPatches = True
TIMING_PAIRS_PER_DATASET = 5  # evenly spaced pairs; compare every method on each pair
WARMUP = 3
REPEATS = 12

set_num_threads(NUMBA_THREADS)
torch.set_num_threads(NUMBA_THREADS)
torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

xfeat = torch.hub.load('verlab/accelerated_features', 'XFeat', pretrained=True, top_k=TOP_K, trust_repo=True).to(device).eval()
for parameter in xfeat.parameters():
    parameter.requires_grad_(False)
print('Device:', device)
# COCO is used for calibration and an internal synthetic-warp check.
# Compression/Blob models use calibration scenes. HPatches is reserved for external evaluation.
DATA_SOURCE = 'coco'  # change to 'upload' to use your own JPG/PNG files
COCO_SOURCE_COUNT = 24
COCO_URL = 'https://images.cocodataset.org/zips/val2017.zip'
drive.mount('/content/drive')
data_dir = Path('/content/drive/MyDrive/Colab Notebooks/CSE527_datasets')
data_dir.mkdir(parents=True, exist_ok=True)
print('Persistent dataset cache:', data_dir)

def download_archive(url, destination):
    if destination.exists() and zipfile.is_zipfile(destination):
        print('Using persistent Drive cache:', destination)
        return
    # Reuse a download from an earlier run in this same Colab session, if present.
    legacy = Path('/content/xfeat_data') / destination.name
    if legacy.exists() and zipfile.is_zipfile(legacy):
        shutil.copy2(legacy, destination)
        print('Copied previous runtime download into Drive:', destination)
        return
    print('Downloading once into Drive:', url)
    subprocess.run(['wget', '-q', '--show-progress', '-c', '-O', str(destination), url], check=True)
    if not zipfile.is_zipfile(destination):
        raise RuntimeError(f'Incomplete or invalid ZIP: {destination}. Delete it and rerun this cell.')

if DATA_SOURCE == 'coco':
    coco_zip = data_dir / 'val2017.zip'
    download_archive(COCO_URL, coco_zip)
    with zipfile.ZipFile(coco_zip) as archive:
        names = sorted(n for n in archive.namelist() if n.lower().endswith(('.jpg', '.jpeg', '.png')))
        if len(names) < COCO_SOURCE_COUNT:
            raise RuntimeError('COCO archive has fewer images than requested.')
        selected = np.random.default_rng(SEED).choice(len(names), COCO_SOURCE_COUNT, replace=False)
        image_items = [(Path(names[i]).name, archive.read(names[i])) for i in selected]
    print(f'Selected {len(image_items)} COCO images using seed {SEED}; no COCO annotations needed.')
elif DATA_SOURCE == 'upload':
    print('Upload JPG/PNG images from distinct scenes.')
    uploaded = files.upload()
    image_items = list(uploaded.items())
    if not image_items:
        raise ValueError('Upload at least one image.')
else:
    raise ValueError("DATA_SOURCE must be 'coco' or 'upload'")

def decode_and_resize(raw_bytes, maximum_side=640):
    image = cv2.imdecode(np.frombuffer(raw_bytes, np.uint8), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError('An uploaded file is not a readable JPG/PNG image.')
    height, width = image.shape[:2]
    scale = min(1.0, maximum_side / max(height, width))
    new_width = max(64, (int(width * scale) // 32) * 32)
    new_height = max(64, (int(height * scale) // 32) * 32)
    return cv2.resize(image, (new_width, new_height))

# Split by source image, before any synthetic warps or descriptor calibration.
# Upload images of distinct scenes; near-duplicate photos belong to the same split.
image_items = sorted(image_items)
images = [decode_and_resize(raw) for _, raw in image_items]
if len(images) >= 2:
    split_rng = np.random.default_rng(SEED)
    order = split_rng.permutation(len(images))
    validation_count = max(1, int(np.ceil(0.25 * len(images))))
    validation_ids = set(order[:validation_count].tolist())
    calibration_images = [im for i, im in enumerate(images) if i not in validation_ids]
    validation_images = [im for i, im in enumerate(images) if i in validation_ids]
    print('Calibration files:', [name for i, (name, _) in enumerate(image_items) if i not in validation_ids])
    print('Held-out files:', [name for i, (name, _) in enumerate(image_items) if i in validation_ids])
else:
    raise ValueError('At least two distinct source images are required.')
print('Calibration images:', len(calibration_images), 'Validation images:', len(validation_images))

Using cache found in /root/.cache/torch/hub/verlab_accelerated_features_main


Device: cpu
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Persistent dataset cache: /content/drive/MyDrive/Colab Notebooks/CSE527_datasets
Using persistent Drive cache: /content/drive/MyDrive/Colab Notebooks/CSE527_datasets/val2017.zip
Selected 24 COCO images using seed 7; no COCO annotations needed.
Calibration files: ['000000002261.jpg', '000000029675.jpg', '000000067534.jpg', '000000162858.jpg', '000000165713.jpg', '000000173008.jpg', '000000175364.jpg', '000000194506.jpg', '000000270402.jpg', '000000289343.jpg', '000000360097.jpg', '000000460841.jpg', '000000471869.jpg', '000000479596.jpg', '000000504711.jpg', '000000517832.jpg', '000000528524.jpg', '000000547383.jpg']
Held-out files: ['000000077595.jpg', '000000132544.jpg', '000000334006.jpg', '000000394559.jpg', '000000449432.jpg', '000000474039.jpg']
Calibration images: 18 Validation images: 6


## 2. 학습용 대응 KP와 고정 BIN 경계
XFeat는 고정합니다. 알려진 warp로 얻은 기하학적 mutual positive를 학습에 사용합니다. 기존 프로토콜처럼 쌍당 최대 500개 positive를 사용합니다. 경계는 calibration 장면에서만 계산합니다.

In [26]:
def image_to_tensor(image):
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb.copy()).permute(2, 0, 1).float()[None].to(device) / 255.0

@torch.no_grad()
def extract_features(image):
    output = xfeat.detectAndCompute(image_to_tensor(image), top_k=TOP_K)[0]
    return {
        'keypoints': output['keypoints'].detach().cpu().float(),
        'descriptors': F.normalize(output['descriptors'].detach().cpu().float(), dim=-1),
    }

def make_random_homography(height, width, generator):
    center = (width / 2.0, height / 2.0)
    angle = generator.uniform(-20.0, 20.0)
    scale = generator.uniform(0.88, 1.08)
    affine = cv2.getRotationMatrix2D(center, angle, scale)
    affine[0, 2] += generator.uniform(-0.06, 0.06) * width
    affine[1, 2] += generator.uniform(-0.06, 0.06) * height
    homography = np.vstack([affine, [0., 0., 1.]]).astype(np.float32)
    homography[2, 0] = generator.uniform(-7e-5, 7e-5)
    homography[2, 1] = generator.uniform(-7e-5, 7e-5)
    return homography

def warp_image(image, homography, generator):
    height, width = image.shape[:2]
    warped = cv2.warpPerspective(image, homography, (width, height),
                                 flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    alpha = float(generator.uniform(0.82, 1.12))
    beta = float(generator.uniform(-12., 18.))
    return np.clip(warped.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)

def project_points(points, homography):
    p = points.detach().cpu().numpy()
    p_h = np.concatenate([p, np.ones((len(p), 1), np.float32)], axis=1)
    projected = (homography @ p_h.T).T
    return torch.from_numpy((projected[:, :2] / projected[:, 2:3]).astype(np.float32))

def ground_truth_pairs(keypoints_1, keypoints_2, homography):
    if not len(keypoints_1) or not len(keypoints_2):
        return torch.empty(0, dtype=torch.long), torch.empty(0, dtype=torch.long)
    distances_12 = torch.cdist(project_points(keypoints_1, homography), keypoints_2)
    nearest_12 = distances_12.argmin(dim=1)
    errors = distances_12[torch.arange(len(keypoints_1)), nearest_12]
    distances_21 = torch.cdist(project_points(keypoints_2, np.linalg.inv(homography)), keypoints_1)
    nearest_21 = distances_21.argmin(dim=1)
    indices = torch.arange(len(keypoints_1))
    good = (nearest_21[nearest_12] == indices) & (errors < GEOMETRIC_THRESHOLD)
    return indices[good], nearest_12[good]

def build_pairs(source_images, warps_per_image, generator):
    pairs = []
    for image_index, image in enumerate(source_images):
        base = extract_features(image)
        for warp_index in range(warps_per_image):
            h = make_random_homography(*image.shape[:2], generator)
            warped = extract_features(warp_image(image, h, generator))
            i, j = ground_truth_pairs(base['keypoints'], warped['keypoints'], h)
            if len(i) > MAX_POSITIVES_PER_PAIR:
                keep = torch.randperm(len(i))[:MAX_POSITIVES_PER_PAIR]
                i, j = i[keep], j[keep]
            if len(i) < 32:
                print(f'Warning: image {image_index}, warp {warp_index}: {len(i)} positive pairs')
            pairs.append({'d1': base['descriptors'], 'd2': warped['descriptors'],
                          'k1': base['keypoints'], 'k2': warped['keypoints'],
                          'pos1': i, 'pos2': j, 'H': h, 'shape1':image.shape[:2], 'shape2':image.shape[:2]})
    return pairs

start = time.perf_counter()
calibration_pairs = build_pairs(calibration_images, CALIBRATION_WARPS_PER_IMAGE, rng)
validation_pairs = build_pairs(validation_images, VALIDATION_WARPS_PER_IMAGE, rng)
calibration = torch.cat([torch.cat([p['d1'], p['d2']], dim=0) for p in calibration_pairs], dim=0)
if len(calibration) > 50000:
    calibration = calibration[torch.randperm(len(calibration))[:50000]]
print('Feature extraction time (calibration and validation):', round(time.perf_counter() - start, 2), 's')
print('Calibration pairs:', len(calibration_pairs), 'Validation pairs:', len(validation_pairs),
      'Calibration descriptors:', len(calibration))

Feature extraction time (calibration and validation): 50.21 s
Calibration pairs: 108 Validation pairs: 18 Calibration descriptors: 50000


In [27]:
# Fixed training-partition quantile edges; no optimizer, projection, or threshold learning.
base_edges_cpu = torch.quantile(calibration, torch.arange(1, LEVELS).float() / LEVELS, dim=0).T.contiguous()

def hard_quantize(descriptors, edges):
    return (descriptors[:, :, None] > edges[None, :, :]).sum(dim=-1).to(torch.uint8)

print('Fixed edges:', tuple(base_edges_cpu.shape), '| learned parameters: 0')


Fixed edges: (64, 7) | learned parameters: 0


## 3. 거리 커널
이 커널들은 앞부분 POPCOUNT 진단 셀에서 이미 정의했습니다.

In [28]:
print("Distance kernels loaded in the first POPCOUNT diagnostic cell.")

Distance kernels loaded in the first POPCOUNT diagnostic cell.


## 4. 두 모델과 공간 Blob 구현
- Grouping: 각 묶음마다 독립적인 8→4→1 tanh 함수. cosine 모델과 uint8 ABS 모델을 따로 학습합니다. 원본 BIN 번호가 아닌 float descriptor에서 시작합니다.
- Blob: 실제 이미지 너비/높이로 grid를 계산합니다. 빈 칸은 제외합니다. Blob 번호는 이미지마다 독립적입니다.
- 학습 Blob은 `64→16`, 평균/최댓값 결합, `32→16` 출력입니다. 내부 KP 순서에 불변입니다.
- 양방향 top-K Blob 후보의 합집합만 비교합니다. 옵션인 인접 칸 확장은 8-neighborhood를 한 번 적용합니다.
- 거리를 계산할 때 선택된 블록만 방문합니다. 모든 KP쌍에 if문을 적용하는 방식이 아닙니다. MNN은 허용 후보 전체에서 결정합니다. 동점은 원래 KP 인덱스가 작은 것을 선택합니다.
- 빈 후보는 unmatched로 남깁니다. 숨겨진 전체 검색 fallback은 없습니다.


In [29]:
import torch.nn as nn
from threadpoolctl import threadpool_limits
import pandas as pd
import matplotlib.pyplot as plt

GROUP_STEPS = 400
BLOB_STEPS = 300
BATCH_SIZE = 128
GRIDS = (4, 6, 8)
TOP_BLOBS = (1, 2, 4)
EXPANSIONS = (False, True)
TARGET_KEEP = .95
TARGET_COST = .20

class GroupScalar(nn.Module):
    """Eight independent 8->4->1 maps; no mixing between groups."""
    def __init__(self):
        super().__init__()
        self.w=nn.Parameter(torch.randn(8,8,4)*.25)
        self.b=nn.Parameter(torch.zeros(8,4))
        self.v=nn.Parameter(torch.randn(8,4)*.25)
        self.c=nn.Parameter(torch.zeros(8))
    def forward(self,x):
        h=torch.relu(torch.einsum('ngi,gih->ngh',x.reshape(-1,8,8),self.w)+self.b)
        return torch.tanh((h*self.v).sum(-1)+self.c)

def group_output(model,x,kind,training=False):
    s=model(x)
    if kind=='cosine': return F.normalize(s,dim=-1,eps=1e-8)
    u=(s+1)*127.5
    if training: return (u+(u.round()-u).detach())/255.
    return u.round().clamp(0,255).to(torch.uint8)

def train_groups(pairs,steps=GROUP_STEPS):
    models={}; logs=[]
    eligible=[p for p in pairs if len(p['pos1'])>=8]
    if not eligible: raise ValueError('No training pairs with enough geometric positives')
    for kind in ('cosine','abs'):
        torch.manual_seed(SEED)
        model=GroupScalar().cpu()
        opt=torch.optim.Adam(model.parameters(),lr=.003)
        gen=np.random.default_rng(SEED)
        start=time.perf_counter()
        for step in range(steps):
            p=eligible[int(gen.integers(len(eligible)))]
            ids=gen.choice(len(p['pos1']),min(BATCH_SIZE,len(p['pos1'])),replace=False)
            a=p['d1'][p['pos1'][ids]]; b=p['d2'][p['pos2'][ids]]
            za=group_output(model,a,kind,True); zb=group_output(model,b,kind,True)
            logits=za@zb.T/.1 if kind=='cosine' else -torch.cdist(za,zb,p=1)/.3
            target=torch.arange(len(a))
            loss=(F.cross_entropy(logits,target)+F.cross_entropy(logits.T,target))/2
            opt.zero_grad(); loss.backward(); opt.step()
            if step%50==0 or step==steps-1:
                logs.append(dict(model='group_'+kind,step=step,loss=float(loss.detach())))
                print('Group',kind,step,'loss',round(float(loss.detach()),4))
        models[kind]=model.eval()
        print('Training seconds',kind,round(time.perf_counter()-start,2))
    return models,logs

def grid_ids(k,shape,g):
    h,w=shape
    xy=np.asarray(k)
    col=np.clip((xy[:,0]*g/w).astype(np.int64),0,g-1)
    row=np.clip((xy[:,1]*g/h).astype(np.int64),0,g-1)
    raw=row*g+col
    occupied,labels=np.unique(raw,return_inverse=True)
    return occupied,labels.astype(np.int64)

class BlobEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.phi=nn.Sequential(nn.Linear(64,16),nn.ReLU())
        self.rho=nn.Linear(32,16)
    def forward(self,x,labels,n):
        h=self.phi(x)
        if n==0: return h.new_empty((0,16))
        # Pooling is permutation-invariant and accepts variable KP counts.
        pooled=[]
        for j in range(n):
            sub=h[labels==j]
            pooled.append(torch.cat([sub.mean(0),sub.amax(0)]))
        return F.normalize(self.rho(torch.stack(pooled)),dim=-1,eps=1e-8)

def blob_ground_truth(p,la,lb,na,nb):
    # Use all geometric mutual positives retained in the training pair.
    positive=torch.zeros((na,nb),dtype=torch.bool)
    positive[la[p['pos1'].numpy()],lb[p['pos2'].numpy()]]=True
    return positive

def multi_positive_loss(scores,positive):
    losses=[]
    for s,p in ((scores,positive),(scores.T,positive.T)):
        valid=p.any(1)
        if valid.any():
            logp=F.log_softmax(s[valid],dim=1)
            target=p[valid].float(); target=target/target.sum(1,keepdim=True)
            losses.append(-(target*logp).sum(1).mean())
    return torch.stack(losses).mean() if losses else None

def train_blob(pairs,steps=BLOB_STEPS):
    torch.manual_seed(SEED+1)
    model=BlobEncoder().cpu(); opt=torch.optim.Adam(model.parameters(),lr=.002)
    gen=np.random.default_rng(SEED+1); logs=[]
    usable=[p for p in pairs if len(p['pos1'])>=8]
    if not usable: raise ValueError('No Blob training positives')
    for step in range(steps):
        p=usable[int(gen.integers(len(usable)))]; g=int(gen.choice(GRIDS))
        oa,la=grid_ids(p['k1'],p['shape1'],g); ob,lb=grid_ids(p['k2'],p['shape2'],g)
        za=model(p['d1'],torch.from_numpy(la),len(oa))
        zb=model(p['d2'],torch.from_numpy(lb),len(ob))
        loss=multi_positive_loss(za@zb.T/.1,blob_ground_truth(p,la,lb,len(oa),len(ob)))
        if loss is None: continue
        opt.zero_grad(); loss.backward(); opt.step()
        if step%50==0 or step==steps-1:
            logs.append(dict(model='blob',step=step,loss=float(loss.detach())))
            print('Blob',step,'loss',round(float(loss.detach()),4))
    return model.eval(),logs

@torch.inference_mode()
def blob_repr(x,k,shape,g,kind,model):
    occupied,labels=grid_ids(k,shape,g)
    if kind=='learned':
        z=model(x,torch.from_numpy(labels),len(occupied)).numpy()
    else:
        z=np.zeros((len(occupied),64),np.float32)
        if len(occupied):
            np.add.at(z,labels,x.numpy())
            z/=np.bincount(labels)[:,None]
            z/=np.maximum(np.linalg.norm(z,axis=1,keepdims=True),1e-8)
    # Group-contiguous KP ordering for block-only descriptor comparisons.
    order=np.argsort(labels,kind='stable').astype(np.int64)
    offsets=np.r_[0,np.cumsum(np.bincount(labels,minlength=len(occupied)))].astype(np.int64)
    return dict(z=z,occupied=occupied,labels=labels,order=order,offsets=offsets)

def select_blobs(a,b,k,g,expand=False):
    scores=a['z']@b['z'].T
    allowed=np.zeros(scores.shape,bool)
    na,nb=scores.shape
    if not na or not nb: return allowed
    ia=np.argsort(-scores,axis=1,kind='stable')[:,:min(k,nb)]
    allowed[np.arange(na)[:,None],ia]=True
    ib=np.argsort(-scores,axis=0,kind='stable')[:min(k,na),:]
    allowed[ib,np.arange(nb)[None,:]]=True
    if expand:
        # Union of both directional spatial dilations; no recursive expansion.
        oa,ob=a['occupied'],b['occupied']
        adj_a=(np.abs(oa[:,None]//g-oa[None,:]//g)<=1)&(np.abs(oa[:,None]%g-oa[None,:]%g)<=1)
        adj_b=(np.abs(ob[:,None]//g-ob[None,:]//g)<=1)&(np.abs(ob[:,None]%g-ob[None,:]%g)<=1)
        allowed=(adj_a.astype(np.int32)@allowed.astype(np.int32)>0)|(allowed.astype(np.int32)@adj_b.astype(np.int32)>0)
    return np.ascontiguousarray(allowed)

@njit(cache=False)
def blob_nearest(pa,pb,oa,ob,sa,sb,allowed):
    n,m=len(pa),len(pb)
    rd=np.full(n,449,np.int64); cd=np.full(m,449,np.int64)
    r=np.full(n,-1,np.int64); c=np.full(m,-1,np.int64)
    count=0
    # Never construct an NxM descriptor distance matrix; visit allowed blocks only.
    for ga in range(allowed.shape[0]):
        for gb in range(allowed.shape[1]):
            if not allowed[ga,gb]: continue
            for ai in range(sa[ga],sa[ga+1]):
                i=oa[ai]
                for bj in range(sb[gb],sb[gb+1]):
                    j=ob[bj]; d=0
                    for k in range(7):
                        d+=np.int64(thermometer_popcount64(pa[i,k]^pb[j,k]))
                    count+=1
                    if d<rd[i] or (d==rd[i] and (r[i]<0 or j<r[i])): rd[i]=d; r[i]=j
                    if d<cd[j] or (d==cd[j] and (c[j]<0 or i<c[j])): cd[j]=d; c[j]=i
    return r,c,count

def mnn(r,c):
    i=np.flatnonzero(r>=0)
    i=i[c[r[i]]==i]
    return i,r[i]

@njit(parallel=True,cache=False)
def abs8(a,b):
    d=np.empty((len(a),len(b)),np.uint16)
    for i in prange(len(a)):
        for j in range(len(b)):
            v=0
            for k in range(8): v+=abs(np.int64(a[i,k])-np.int64(b[j,k]))
            d[i,j]=v
    return d

def test_new_kernels():
    gen=np.random.default_rng(3)
    for n,m in [(0,0),(0,8),(8,0),(1,1),(17,23)]:
        a=gen.integers(0,8,(n,64),dtype=np.uint8); b=gen.integers(0,8,(m,64),dtype=np.uint8)
        pa,pb=pack_thermometer64(a),pack_thermometer64(b)
        la=np.arange(n)%3; lb=np.arange(m)%4
        oa=np.argsort(la).astype(np.int64); ob=np.argsort(lb).astype(np.int64)
        sa=np.r_[0,np.cumsum(np.bincount(la,minlength=3))].astype(np.int64)
        sb=np.r_[0,np.cumsum(np.bincount(lb,minlength=4))].astype(np.int64)
        for allowed in [np.ones((3,4),bool),gen.random((3,4))>.5,np.zeros((3,4),bool)]:
            r,c,count=blob_nearest(pa,pb,oa,ob,sa,sb,allowed)
            d=pairwise_uint8_abs(a,b).astype(np.int64)
            mask=allowed[la[:,None],lb[None,:]]
            assert count==int(mask.sum())
            d[~mask]=9999
            er=np.full(n,-1,np.int64); ec=np.full(m,-1,np.int64)
            if n and m:
                er=d.argmin(1); er[~mask.any(1)]=-1
                ec=d.argmin(0); ec[~mask.any(0)]=-1
            np.testing.assert_array_equal(r,er); np.testing.assert_array_equal(c,ec)
    # Ties with reversed within-blob order must still choose lowest original index.
    z=pack_thermometer64(np.zeros((5,64),np.uint8)); order=np.arange(4,-1,-1,dtype=np.int64)
    r,c,_=blob_nearest(z,z,order,order,np.array([0,5]),np.array([0,5]),np.ones((1,1),bool))
    assert np.all(r==0) and np.all(c==0)
    a=gen.integers(0,256,(11,8),dtype=np.uint8); b=gen.integers(0,256,(13,8),dtype=np.uint8)
    np.testing.assert_array_equal(abs8(a,b),np.abs(a.astype(int)[:,None]-b.astype(int)[None,:]).sum(2))
    model=GroupScalar(); x=torch.randn(10,64)
    assert model(x).shape==(10,8)
    assert group_output(model,x,'abs').dtype==torch.uint8
    model2=BlobEncoder(); labels=torch.arange(10)%3; perm=torch.randperm(10)
    torch.testing.assert_close(model2(x,labels,3),model2(x[perm],labels[perm],3),atol=1e-6,rtol=1e-5)
    print('PASS: sparse blocks vs masked exhaustive reference, empty candidates, ties, uint8 overflow, grouping dimensions, Blob permutation invariance.')

test_new_kernels()


PASS: sparse blocks vs masked exhaustive reference, empty candidates, ties, uint8 overflow, grouping dimensions, Blob permutation invariance.


**기본 실행에서는 건너뜁니다. `RUN_LEGACY_EXPERIMENTS=True`일 때만 실행.**

## 5. 압축기와 Blob 학습
기본 Grouping 각 400 steps, Blob 300 steps. 처음 학습 budget이며 최적 성능을 보장하지 않습니다. 학습 손실과 checkpoint를 저장합니다. HPatches는 학습에 사용하지 않습니다.

In [30]:
if RUN_LEGACY_EXPERIMENTS:
    # 1. 먼저 추론용 텐서를 일반 텐서로 복사
    with torch.inference_mode(False):
        for pair in calibration_pairs:
            for key in ("d1", "d2"):
                pair[key] = pair[key].detach().cpu().clone()

    # 2. 그다음 학습
    train_start = time.perf_counter()

    group_models, group_logs = train_groups(calibration_pairs)
    blob_model, blob_logs = train_blob(calibration_pairs)

    training_seconds = time.perf_counter() - train_start
    training_log = pd.DataFrame(group_logs + blob_logs)

    display(training_log)
    print("Total model training seconds:", training_seconds)
else:
    print("Legacy experiment skipped. Continue to optimized comparison below.")


Legacy experiment skipped. Continue to optimized comparison below.


## 6. HPatches 평가 데이터 준비
6 viewpoint + 6 illumination 시퀀스, 1→2/1→3의 24쌍. resize에 맞춰 homography를 변환합니다.

In [31]:
real_pairs = []
if RUN_HPatches:
    HPATCHES_URL = 'https://huggingface.co/datasets/vbalnt/hpatches/resolve/main/hpatches-sequences-release.zip'
    HPATCHES_SEQUENCES_PER_TYPE = 6  # 6 viewpoint + 6 illumination scenes
    HPATCHES_TARGETS = (2, 3)       # two real photos per sequence
    hpatches_zip = data_dir / 'hpatches-sequences-release.zip'
    download_archive(HPATCHES_URL, hpatches_zip)

    def decode_hpatches_with_scale(raw_bytes):
        original = cv2.imdecode(np.frombuffer(raw_bytes, np.uint8), cv2.IMREAD_COLOR)
        if original is None:
            raise RuntimeError('Failed to decode HPatches PPM image')
        old_h, old_w = original.shape[:2]
        resized = decode_and_resize(raw_bytes)
        new_h, new_w = resized.shape[:2]
        scale = np.diag([new_w / old_w, new_h / old_h, 1.0]).astype(np.float64)
        return resized, scale

    real_pairs = []
    with zipfile.ZipFile(hpatches_zip) as archive:
        entries = set(archive.namelist())
        folders = sorted({name.rsplit('/', 1)[0] for name in entries if name.endswith('/1.ppm')})
        for prefix in ('v_', 'i_'):
            selected_folders = [folder for folder in folders if folder.split('/')[-1].startswith(prefix)]
            if len(selected_folders) < HPATCHES_SEQUENCES_PER_TYPE:
                raise RuntimeError(f'Expected more HPatches {prefix} folders; found {len(selected_folders)}')
            for folder in selected_folders[:HPATCHES_SEQUENCES_PER_TYPE]:
                image_1, scale_1 = decode_hpatches_with_scale(archive.read(folder + '/1.ppm'))
                features_1 = extract_features(image_1)
                for target_id in HPATCHES_TARGETS:
                    image_name = f'{folder}/{target_id}.ppm'
                    h_name = f'{folder}/H_1_{target_id}'
                    if image_name not in entries or h_name not in entries:
                        raise RuntimeError(f'Missing HPatches image or homography: {folder}, {target_id}')
                    image_2, scale_2 = decode_hpatches_with_scale(archive.read(image_name))
                    features_2 = extract_features(image_2)
                    original_h = np.loadtxt(io.BytesIO(archive.read(h_name))).reshape(3, 3)
                    resized_h = scale_2 @ original_h @ np.linalg.inv(scale_1)
                    real_pairs.append({
                        'name': f'{folder.split("/")[-1]}:1->{target_id}',
                        'group': prefix, 'd1': features_1['descriptors'],
                        'd2': features_2['descriptors'],
                        'k1': features_1['keypoints'], 'k2': features_2['keypoints'],
                        'H': resized_h, 'shape1': image_1.shape[:2], 'shape2': image_2.shape[:2],
                    })
    print('HPatches real-photo pairs:', len(real_pairs), '(viewpoint and illumination)')



Using persistent Drive cache: /content/drive/MyDrive/Colab Notebooks/CSE527_datasets/hpatches-sequences-release.zip
HPatches real-photo pairs: 24 (viewpoint and illumination)


**기본 실행에서는 건너뜁니다. `RUN_LEGACY_EXPERIMENTS=True`일 때만 실행.**

## 7. Blob 설정 선택 → 정확도·속도 측정

1. 합성 **검증 장면**에서 grid=4/6/8, top-K=1/2/4, 인접 칸 확장 유무를 비교합니다.
2. 기존 full Thermometer의 올바른 매칭을 95% 이상 후보에 남기는 설정 중 비교쌍이 가장 적은 것을 선택합니다. 없으면 유지율이 가장 높은 설정을 택하고 목표 미달을 출력합니다. 비교쌍 20% 이하 목표의 달성 여부도 출력합니다.
3. mean/learned 표현별 설정을 고정하여 HPatches를 평가합니다. 합성 결과는 설정 선택에 사용된 validation 수치이며 독립 test 결과가 아닙니다.
4. warmup=3, repeats=12, 데이터셋당 5쌍. 방법·측정 순서를 섞습니다. JIT/검증/학습/XFeat/I/O는 매칭 시간에서 제외합니다.

**출력:** pooled precision@3px, correct/mutual, 전체 매칭의 median error, 후보 비교 비율, reference 정답 후보 유지율, cached match / prepare / total 시간.
- `prepare_ms`: Grouping 압축 또는 BIN 인코딩·packing·Blob 생성·표현.
- `cached_match_ms`: 준비된 표현에서 시작. Blob 검색·후보 선택·내부 매칭까지 포함.
- `total_ms`: float XFeat descriptor와 좌표에서 시작하는 전체 준비+매칭을 직접 측정.
- 시간은 pair별 반복 중앙값을 구한 뒤 pair 중앙값. 전체 시간과 분리 시간의 합은 반복 잡음 때문에 다를 수 있습니다.
- 3px는 resize된 이미지 좌표이며 공식 HPatches descriptor benchmark가 아닙니다.


In [32]:
if RUN_LEGACY_EXPERIMENTS:
    EDGES=np.ascontiguousarray(base_edges_cpu.numpy(),dtype=np.float32)
    def bins(x): return np.ascontiguousarray((x.numpy()[:,:,None]>EDGES[None]).sum(2),dtype=np.uint8)
    def errors_for(p,ij):
        i,j=ij
        return np.linalg.norm(project_points(p['k1'][i],p['H']).numpy()-p['k2'][j].numpy(),axis=1)

    def reference_correct(p):
        # Untimed evaluation oracle only; never used by Blob candidate selection.
        a,b=bins(p['d1']),bins(p['d2'])
        ij=mutual_from_scores(pairwise_thermometer64(pack_thermometer64(a),pack_thermometer64(b)),False)
        ok=errors_for(p,ij)<GEOMETRIC_THRESHOLD
        return ij[0][ok],ij[1][ok]

    def candidate_stats(a,b,allowed,reference):
        counts_a=np.diff(a['offsets']); counts_b=np.diff(b['offsets'])
        evaluated=int((allowed*counts_a[:,None]*counts_b[None,:]).sum())
        total=int(counts_a.sum()*counts_b.sum())
        i,j=reference
        kept=int(allowed[a['labels'][i],b['labels'][j]].sum())
        return evaluated,total,kept,len(i)

    # Tune only on held-out synthetic source scenes; HPatches is untouched until final evaluation.
    sweep=[]
    val_references=[reference_correct(p) for p in validation_pairs]
    for kind in ('mean','learned'):
        for g in GRIDS:
            representations=[(blob_repr(p['d1'],p['k1'],p['shape1'],g,kind,blob_model),
                              blob_repr(p['d2'],p['k2'],p['shape2'],g,kind,blob_model)) for p in validation_pairs]
            for k in TOP_BLOBS:
                for expand in EXPANSIONS:
                    evaluated=total=kept=correct=0
                    for (a,b),ref in zip(representations,val_references):
                        allowed=select_blobs(a,b,k,g,expand)
                        e,t,c,r=candidate_stats(a,b,allowed,ref)
                        evaluated+=e; total+=t; kept+=c; correct+=r
                    sweep.append(dict(kind=kind,grid=g,k=k,expand=expand,
                        candidate_fraction=evaluated/total if total else 0,
                        reference_correct_keep=kept/correct if correct else np.nan,
                        reference_correct=correct))
    sweep_df=pd.DataFrame(sweep)
    configs={}
    for kind in ('mean','learned'):
        rows=sweep_df[sweep_df.kind==kind]
        valid=rows[rows.reference_correct_keep>=TARGET_KEEP]
        if len(valid):
            chosen=valid.sort_values(['candidate_fraction','reference_correct_keep'],ascending=[True,False]).iloc[0]
        else:
            chosen=rows.sort_values(['reference_correct_keep','candidate_fraction'],ascending=[False,True]).iloc[0]
        configs[kind]=dict(grid=int(chosen.grid),k=int(chosen.k),expand=bool(chosen.expand))
        print('Frozen Blob config',kind,configs[kind],
              'validation keep',round(chosen.reference_correct_keep,4),
              'candidate fraction',round(chosen.candidate_fraction,4),
              '95% / 20% target met:',bool(chosen.reference_correct_keep>=TARGET_KEEP and chosen.candidate_fraction<=TARGET_COST))
    display(sweep_df.round(4))

    METHODS=['group8_cosine','group8_uint8_abs','blob_mean','blob_learned']
    @torch.inference_mode()
    def prepare_new(p,method):
        if method.startswith('group8'):
            kind='cosine' if method=='group8_cosine' else 'abs'
            return dict(a=np.ascontiguousarray(group_output(group_models[kind],p['d1'],kind).numpy()),
                        b=np.ascontiguousarray(group_output(group_models[kind],p['d2'],kind).numpy()))
        kind=method.split('_',1)[1]; cfg=configs[kind]; g=cfg['grid']
        return dict(a=blob_repr(p['d1'],p['k1'],p['shape1'],g,kind,blob_model),
                    b=blob_repr(p['d2'],p['k2'],p['shape2'],g,kind,blob_model),
                    pa=pack_thermometer64(bins(p['d1'])),pb=pack_thermometer64(bins(p['d2'])),cfg=cfg)

    def match_new(z,method):
        if method.startswith('group8'):
            if method=='group8_cosine': ij=mutual_from_scores(z['a']@z['b'].T,True)
            else: ij=mutual_from_scores(abs8(z['a'],z['b']),False)
            return ij, len(z['a'])*len(z['b']),None
        a,b=z['a'],z['b']; cfg=z['cfg']
        allowed=select_blobs(a,b,cfg['k'],cfg['grid'],cfg['expand'])
        r,c,count=blob_nearest(z['pa'],z['pb'],a['order'],b['order'],a['offsets'],b['offsets'],allowed)
        return mnn(r,c),count,allowed

    def full_new(p,method): return match_new(prepare_new(p,method),method)

    datasets={'synthetic_validation':validation_pairs}
    if real_pairs: datasets['HPatches_test']=real_pairs
    accuracy_rows=[]; timing_rows=[]; all_errors={}; oracle_rows=[]
    def elapsed(fn):
        t=time.perf_counter_ns(); fn(); return (time.perf_counter_ns()-t)/1e6
    with threadpool_limits(limits=NUMBA_THREADS):
        for dataset,pairs in datasets.items():
            timing_ids=set(np.unique(np.linspace(0,len(pairs)-1,min(TIMING_PAIRS_PER_DATASET,len(pairs)),dtype=int)))
            for pi,p in enumerate(pairs):
                ref=reference_correct(p)
                oracle_rows.append(dict(dataset=dataset,pair=pi,full_thermometer_correct=len(ref[0])))
                cached={method:prepare_new(p,method) for method in METHODS}
                for method in METHODS:
                    z=cached[method]; ij,count,allowed=match_new(z,method)
                    err=errors_for(p,ij); all_errors.setdefault((dataset,method),[]).extend(err.tolist())
                    n,m=len(p['d1']),len(p['d2']); total=n*m
                    kept=len(ref[0])
                    if allowed is not None:
                        e,t,kept,r=candidate_stats(z['a'],z['b'],allowed,ref)
                        assert e==count and t==total
                        # Validate sparse result against masked exhaustive distance once per dataset/method.
                        if pi==0 and n and m:
                            matrix=pairwise_thermometer64(z['pa'],z['pb']).astype(np.int64)
                            mask=allowed[z['a']['labels'][:,None],z['b']['labels'][None,:]]
                            matrix[~mask]=9999
                            rr=matrix.argmin(1); cc=matrix.argmin(0)
                            rr[~mask.any(1)]=-1; cc[~mask.any(0)]=-1
                            expected=mnn(rr,cc)
                            for a,b in zip(expected,ij): np.testing.assert_array_equal(a,b)
                    accuracy_rows.append(dict(dataset=dataset,pair=pi,pair_name=p.get('name',str(pi)),
                        method=method,n=n,m=m,mutual=len(ij[0]),correct=int((err<GEOMETRIC_THRESHOLD).sum()),
                        candidate_pairs=count,total_pairs=total,reference_correct=len(ref[0]),reference_kept=kept))
                if pi in timing_ids and len(p['d1']) and len(p['d2']):
                    for method in METHODS:
                        expected=match_new(cached[method],method)[0]
                        actual=full_new(p,method)[0]
                        for a,b in zip(expected,actual):np.testing.assert_array_equal(a,b)
                        for _ in range(WARMUP):
                            match_new(cached[method],method);full_new(p,method);prepare_new(p,method)
                    rng_t=np.random.default_rng(SEED+pi)
                    for repeat in range(REPEATS):
                        for method in rng_t.permutation(METHODS):
                            fns={'cached_match_ms':lambda:match_new(cached[method],method),
                                 'prepare_ms':lambda:prepare_new(p,method),
                                 'total_ms':lambda:full_new(p,method)}
                            times={phase:elapsed(fns[phase]) for phase in rng_t.permutation(list(fns))}
                            timing_rows.append(dict(dataset=dataset,pair=pi,method=str(method),repeat=repeat,**times))
                print('Evaluated',dataset,pi+1,'/',len(pairs))
    accuracy=pd.DataFrame(accuracy_rows); timing_raw=pd.DataFrame(timing_rows)
    summary_rows=[]
    for (dataset,method),df in accuracy.groupby(['dataset','method'],sort=False):
        err=all_errors[dataset,method];mutual=int(df.mutual.sum());correct=int(df.correct.sum())
        ref=int(df.reference_correct.sum())
        summary_rows.append(dict(dataset=dataset,method=method,mutual=mutual,correct=correct,
            precision_pct=100*correct/mutual if mutual else np.nan,median_px=np.median(err) if err else np.nan,
            candidate_pct=100*df.candidate_pairs.sum()/df.total_pairs.sum(),
            reference_correct_keep_pct=100*df.reference_kept.sum()/ref if ref else np.nan,
            descriptor_bytes=32 if method=='group8_cosine' else 8 if method=='group8_uint8_abs' else 56))
    summary=pd.DataFrame(summary_rows)
    if len(timing_raw):
        timing_pair=timing_raw.groupby(['dataset','pair','method'])[['cached_match_ms','prepare_ms','total_ms']].median().reset_index()
        timing_summary=timing_pair.groupby(['dataset','method'])[['cached_match_ms','prepare_ms','total_ms']].median().reset_index()
        summary=summary.merge(timing_summary,on=['dataset','method'],how='left')
    else:
        timing_pair=pd.DataFrame()
        for col in ('cached_match_ms','prepare_ms','total_ms'):summary[col]=np.nan
    display(summary.round(4))
    print('reference_correct_keep_pct is candidate survival, not final-match recall. Grouping searches all pairs so its candidate survival is 100%.')
    print('Blob descriptor_bytes excludes its extra representation (mean: 256B/blob, learned:64B/blob), membership/index buffers and model weights.')
    print('Synthetic validation selected Blob settings; HPatches_test is the external evaluation.')
    print('Times exclude feature extraction and training. prepare includes compression/Blob construction; cached_match includes Blob search and block matching.')
    print('No old latency benchmarks are rerun. Do not infer speedup by mixing timings from different Colab machines.')
    fig,axes=plt.subplots(len(datasets),3,figsize=(17,4.5*len(datasets)),squeeze=False)
    for row,dataset in enumerate(datasets):
        df=summary[summary.dataset==dataset].set_index('method').reindex(METHODS)
        df[['cached_match_ms','total_ms']].plot.barh(ax=axes[row,0],title=dataset+' time (ms)')
        df[['precision_pct']].plot.barh(ax=axes[row,1],title='Precision @ 3px',legend=False)
        axes[row,1].set_xlim(0,100)
        df[['candidate_pct','reference_correct_keep_pct']].plot.barh(ax=axes[row,2],title='Candidate cost / survival (%)')
        axes[row,2].set_xlim(0,105)
    fig.tight_layout();display(fig);plt.close(fig)
else:
    print("Legacy experiment skipped. Continue to optimized comparison below.")


Legacy experiment skipped. Continue to optimized comparison below.


**기본 실행에서는 건너뜁니다. `RUN_LEGACY_EXPERIMENTS=True`일 때만 실행.**

## 8. 모델·설정·결과 저장
기존 파일을 덮어쓰지 않고 새로운 서울시각 폴더에 저장합니다. 마지막 출력의 RESULT FOLDER에서 CSV/JSON/그래프/checkpoint를 확인하세요.

In [33]:
if RUN_LEGACY_EXPERIMENTS:
    import json,platform,hashlib,numba
    from datetime import datetime
    from zoneinfo import ZoneInfo
    from threadpoolctl import threadpool_info
    run_id=datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S_%f')
    result_dir=data_dir/'experiment_results'/('learned_grouping_blob_'+run_id)
    result_dir.mkdir(parents=True,exist_ok=False)
    for name,df in [('summary.csv',summary),('accuracy_per_pair.csv',accuracy),
                    ('timing_raw.csv',timing_raw),('timing_per_pair.csv',timing_pair),
                    ('blob_validation_sweep.csv',sweep_df),('training_loss.csv',training_log)]:
        df.to_csv(result_dir/name,index=False)
    fig.savefig(result_dir/'comparison.png',dpi=160,bbox_inches='tight')
    torch.save({'group_cosine':group_models['cosine'].state_dict(),
                'group_abs':group_models['abs'].state_dict(),
                'blob':blob_model.state_dict(),'edges':base_edges_cpu,
                'blob_configs':configs,'seed':SEED},result_dir/'models_and_config.pt')
    def records(df):return json.loads(df.to_json(orient='records'))
    report={'experiment':'learned grouping and spatial blob only','run_id_seoul':run_id,
            'seed':SEED,'top_k':TOP_K,'threshold_resized_px':GEOMETRIC_THRESHOLD,
            'group_steps':GROUP_STEPS,'blob_steps':BLOB_STEPS,'batch_size':BATCH_SIZE,
            'training_seconds':training_seconds,'blob_configs':configs,
            'warmup':WARMUP,'repeats':REPEATS,'timing_pairs_per_dataset':TIMING_PAIRS_PER_DATASET,
            'cpu_threads':NUMBA_THREADS,'blob_kernel_threads':1,'extraction_device':device,
            'platform':platform.platform(),'versions':{'numpy':np.__version__,'torch':torch.__version__,'numba':numba.__version__},
            'threadpools':threadpool_info(),
            'training_sources':[name for i,(name,_) in enumerate(image_items) if i not in validation_ids],
            'validation_sources':[name for i,(name,_) in enumerate(image_items) if i in validation_ids],
            'edges_sha256':hashlib.sha256(EDGES.tobytes()).hexdigest(),
            'summary':records(summary),'validation_sweep':records(sweep_df),
            'accuracy_per_pair':records(accuracy),'reference_counts':oracle_rows,
            'notes':['No legacy latency benchmarks rerun.',
                     'Reference correct retention means candidate survival, not final recall.',
                     'Grouping full search; Blob exact only within selected candidate graph.',
                     'Models frozen before validation sweep; configurations frozen before HPatches test.',
                     'Blob storage adds vectors, membership/order arrays and model weights.']}
    with open(result_dir/'results.json','w') as f:json.dump(report,f,indent=2,allow_nan=False)
    print('RESULT FOLDER:',result_dir)
    for path in sorted(result_dir.iterdir()):print(path.name)
else:
    print("Legacy experiment skipped. Continue to optimized comparison below.")


Legacy experiment skipped. Continue to optimized comparison below.


## 9. 통합 인코딩·병렬 Blob 커널
직접 인코딩은 `x > threshold`를 그대로 사용합니다. 동일/중복 경계에서도 기존 BIN과 같은 비트를 만듭니다.

병렬 작업은 A의 Blob별로 분리하고, 각 작업의 B측 최근접 후보를 별도 보관한 뒤 최종 병합합니다. shared array write race 없이 최소 원래 KP index로 동점을 처리합니다. 추가 column buffer는 `A Blob 수 × B KP 수 × 10 bytes`입니다.

기존 float 평균의 계산 순서는 유지합니다. BIN 평균 실험은 encoding과 그룹 합 누적을 하나의 커널로 합치며, Blob 간 평균 BIN L1로 후보를 고릅니다. 서로 다른 평균을 사용하는 이 실험에는 동일 MNN을 요구하지 않습니다.


In [34]:
# CPU kernels: sorted quantile edges, 64 dimensions, seven thermometer planes.
@njit(parallel=True, cache=False)
def direct_thermo(x, edges):
    out=np.zeros((len(x),7),np.uint64)
    for i in prange(len(x)):
        for d in range(64):
            bit=np.uint64(1)<<np.uint64(d)
            for k in range(7):
                if x[i,d]>edges[d,k]: out[i,k]|=bit
    return out

@njit(parallel=True, cache=False)
def direct_thermo_binmean(x,edges,order,offsets):
    # Each blob owns its output rows and its sum; no atomic/shared writes.
    out=np.zeros((len(x),7),np.uint64)
    sums=np.zeros((len(offsets)-1,64),np.float32)
    for g in prange(len(offsets)-1):
        for pos in range(offsets[g],offsets[g+1]):
            i=order[pos]
            for d in range(64):
                bit=np.uint64(1)<<np.uint64(d); q=0
                for k in range(7):
                    if x[i,d]>edges[d,k]:out[i,k]|=bit;q+=1
                sums[g,d]+=q
        count=offsets[g+1]-offsets[g]
        if count:
            for d in range(64):sums[g,d]/=count
    return out,sums

@njit(parallel=True,cache=False)
def parallel_blob_nearest(pa,pb,oa,ob,sa,sb,allowed):
    n,m=len(pa),len(pb); ng=len(sa)-1
    r=np.full(n,-1,np.int64); rd=np.full(n,449,np.uint16)
    # Per-A-blob column winners avoid races between workers.
    local_d=np.full((ng,m),449,np.uint16)
    local_i=np.full((ng,m),-1,np.int64)
    counts=np.zeros(ng,np.int64)
    for ga in prange(ng):
        for gb in range(len(sb)-1):
            if not allowed[ga,gb]:continue
            for ai in range(sa[ga],sa[ga+1]):
                i=oa[ai]
                for bj in range(sb[gb],sb[gb+1]):
                    j=ob[bj];d=0
                    for k in range(7):d+=np.int64(thermometer_popcount64(pa[i,k]^pb[j,k]))
                    counts[ga]+=1
                    if d<rd[i] or (d==rd[i] and (r[i]<0 or j<r[i])):rd[i]=d;r[i]=j
                    if d<local_d[ga,j] or (d==local_d[ga,j] and (local_i[ga,j]<0 or i<local_i[ga,j])):
                        local_d[ga,j]=d;local_i[ga,j]=i
    c=np.full(m,-1,np.int64)
    for j in prange(m):
        best=449;idx=-1
        for ga in range(ng):
            d=local_d[ga,j];i=local_i[ga,j]
            if i>=0 and (d<best or (d==best and (idx<0 or i<idx))):best=d;idx=i
        c[j]=idx
    return r,c,counts.sum()

def grid_layout(k,shape,g):
    h,w=shape
    xy=np.asarray(k)
    if h<=0 or w<=0:raise ValueError('Image shape must be positive')
    raw=np.clip((xy[:,1]*g/h).astype(np.int64),0,g-1)*g+np.clip((xy[:,0]*g/w).astype(np.int64),0,g-1)
    occupied,labels=np.unique(raw,return_inverse=True)
    order=np.argsort(labels,kind='stable').astype(np.int64)
    offsets=np.r_[0,np.cumsum(np.bincount(labels,minlength=len(occupied)))].astype(np.int64)
    return dict(occupied=occupied,labels=labels.astype(np.int64),order=order,offsets=offsets)

def original_float_mean(x,layout):
    # Preserve the former mean's exact operations/order for candidate equivalence.
    z=np.zeros((len(layout['occupied']),64),np.float32)
    if len(z):
        np.add.at(z,layout['labels'],x)
        z/=np.bincount(layout['labels'])[:,None]
        z/=np.maximum(np.linalg.norm(z,axis=1,keepdims=True),1e-8)
    return z

def blob_candidate_graph(a,b,k,g,kind='float_mean'):
    if kind=='bin_mean':
        distances=np.abs(a['z'][:,None,:]-b['z'][None,:,:]).sum(2)
        score=-distances
    else:score=a['z']@b['z'].T
    allowed=np.zeros(score.shape,bool);na,nb=score.shape
    if na and nb:
        ix=np.argsort(-score,axis=1,kind='stable')[:,:min(k,nb)]
        allowed[np.arange(na)[:,None],ix]=True
        ix=np.argsort(-score,axis=0,kind='stable')[:min(k,na),:]
        allowed[ix,np.arange(nb)[None,:]]=True
    return np.ascontiguousarray(allowed)

def quantize_array(x,edges):
    return np.ascontiguousarray((x[:,:,None]>edges[None,:,:]).sum(2),dtype=np.uint8)

def assert_optimized_kernels():
    gen=np.random.default_rng(9)
    edges=np.sort(gen.normal(size=(64,7)).astype(np.float32),axis=1)
    edges[0,1:4]=edges[0,1] # repeated thresholds
    for n,m in [(0,0),(0,7),(7,0),(1,1),(17,23),(64,71)]:
        x=gen.normal(size=(n,64)).astype(np.float32);y=gen.normal(size=(m,64)).astype(np.float32)
        if n:x[0]=edges[:,0] # exact edge: strict >, not >=
        a=grid_layout(gen.uniform(0,100,(n,2)),(100,100),4)
        b=grid_layout(gen.uniform(0,100,(m,2)),(100,100),4)
        qa,qb=quantize_array(x,edges),quantize_array(y,edges)
        pa,pb=direct_thermo(x,edges),direct_thermo(y,edges)
        np.testing.assert_array_equal(pa,pack_thermometer64(qa))
        np.testing.assert_array_equal(pb,pack_thermometer64(qb))
        fa,means=direct_thermo_binmean(x,edges,a['order'],a['offsets'])
        np.testing.assert_array_equal(fa,pa)
        for g in range(len(means)):
            np.testing.assert_allclose(means[g],qa[a['labels']==g].mean(0),atol=1e-6)
        for mode in range(3):
            allowed=gen.random((len(a['occupied']),len(b['occupied'])))>.5
            if mode==0:allowed[:]=True
            if mode==1:allowed[:]=False
            args=(pa,pb,a['order'],b['order'],a['offsets'],b['offsets'],allowed)
            sr,sc,sn=blob_nearest(*args);r,c,count=parallel_blob_nearest(*args)
            np.testing.assert_array_equal(r,sr);np.testing.assert_array_equal(c,sc);assert count==sn
            dense=pairwise_thermometer64(pa,pb).astype(np.int64)
            mask=allowed[a['labels'][:,None],b['labels'][None,:]]
            dense[~mask]=9999
            er=np.full(n,-1,np.int64);ec=np.full(m,-1,np.int64)
            if n and m:
                er=dense.argmin(1);ec=dense.argmin(0)
                er[~mask.any(1)]=-1;ec[~mask.any(0)]=-1
            np.testing.assert_array_equal(r,er);np.testing.assert_array_equal(c,ec)
            assert count==mask.sum()
    # Tied matches in different A blobs, and unsorted original KP indices.
    p=np.zeros((8,7),np.uint64);o=np.array([7,5,3,1,6,4,2,0],np.int64)
    r,c,count=parallel_blob_nearest(p,p,o,o,np.array([0,4,8]),np.array([0,4,8]),np.ones((2,2),bool))
    assert np.all(r==0) and np.all(c==0) and count==64
    print('PASS: direct encoding bits, repeated/equal boundaries, BIN means, serial/parallel NN equivalence, masked exhaustive oracle, empty sets/candidates and cross-blob ties.')

if RUN_BLOB_BENCHMARKS:
    assert_optimized_kernels()


## 10. 같은 실행에서 정확도·시간 비교
7개 방법: float cosine / 기존 split Thermometer / direct Thermometer / 기존 평균 Blob serial / direct 평균 Blob serial / direct 평균 Blob parallel / direct BIN평균 Blob parallel.

모든 실제 평가 쌍에서 exact 변경의 bit·양방향 NN·MNN 일치를 검증합니다. 기존 평균 Blob의 후보 수를 유지하여 병렬화 효과와 후보 변경 효과를 분리합니다.

캐시 매칭은 Blob 후보 검색부터 포함합니다. total은 두 float descriptor 집합에서 인코딩·Blob 준비·매칭까지 직접 측정합니다. 세부 단계 중앙값은 진단용이며 합이 total과 같을 필요가 없습니다. 모든 방식에서 XFeat 추출·JIT·검증·파일저장은 제외합니다. BLAS/Numba/Torch는 설정한 CPU threads로 제한합니다.


In [35]:
if RUN_BLOB_BENCHMARKS:
    import time,json,platform,hashlib
    import pandas as pd
    import matplotlib.pyplot as plt
    from threadpoolctl import threadpool_limits,threadpool_info

    # Fixed to the user's previous selected mean-Blob config; no HPatches tuning.
    OPT_GRID=6
    OPT_TOP_K=4
    OPT_METHODS=['float_cosine','thermo_split','thermo_direct',
                 'blob_mean_split_serial','blob_mean_direct_serial',
                 'blob_mean_direct_parallel','blob_binmean_direct_parallel']

    def as_array(t):
        return np.ascontiguousarray(t.detach().cpu().numpy() if hasattr(t,'detach') else t,dtype=np.float32)

    OPT_EDGES=as_array(base_edges_cpu)
    assert OPT_EDGES.shape==(64,7) and np.all(np.diff(OPT_EDGES,axis=1)>=0)

    def opt_pair(p):
        return dict(x=as_array(p['d1']),y=as_array(p['d2']),
                    k1=as_array(p['k1']),k2=as_array(p['k2']),H=np.asarray(p['H']),
                    shape1=p['shape1'],shape2=p['shape2'],name=p.get('name',''))

    def layouts(p):
        return (grid_layout(p['k1'],p['shape1'],OPT_GRID),grid_layout(p['k2'],p['shape2'],OPT_GRID))

    def opt_prepare(p,method):
        if method=='float_cosine':return dict(x=p['x'],y=p['y'])
        use_blob=method.startswith('blob_'); binmean='binmean' in method
        if use_blob:a,b=layouts(p)
        if binmean:
            pa,a['z']=direct_thermo_binmean(p['x'],OPT_EDGES,a['order'],a['offsets'])
            pb,b['z']=direct_thermo_binmean(p['y'],OPT_EDGES,b['order'],b['offsets'])
        else:
            if 'split' in method:
                pa=pack_thermometer64(quantize_array(p['x'],OPT_EDGES))
                pb=pack_thermometer64(quantize_array(p['y'],OPT_EDGES))
            else:pa,pb=direct_thermo(p['x'],OPT_EDGES),direct_thermo(p['y'],OPT_EDGES)
            if use_blob:a['z']=original_float_mean(p['x'],a);b['z']=original_float_mean(p['y'],b)
        z=dict(pa=pa,pb=pb)
        if use_blob:z.update(a=a,b=b)
        return z

    def opt_select(z,method):
        return blob_candidate_graph(z['a'],z['b'],OPT_TOP_K,OPT_GRID,'bin_mean' if 'binmean' in method else 'float_mean')

    def opt_kernel(z,allowed,method):
        a,b=z['a'],z['b']
        fn=parallel_blob_nearest if method.endswith('parallel') else blob_nearest
        return fn(z['pa'],z['pb'],a['order'],b['order'],a['offsets'],b['offsets'],allowed)

    def opt_match(z,method):
        if method=='float_cosine':return mutual_from_scores(z['x']@z['y'].T,True)
        if method.startswith('thermo_'):return mutual_from_scores(pairwise_thermometer64(z['pa'],z['pb']),False)
        r,c,_=opt_kernel(z,opt_select(z,method),method)
        return mnn(r,c)

    def opt_total(p,method):return opt_match(opt_prepare(p,method),method)

    def opt_errors(p,ij):
        i,j=ij
        homogeneous=np.c_[p['k1'][i],np.ones(len(i))]@p['H'].T
        with np.errstate(divide='ignore',invalid='ignore'):
            e=np.linalg.norm(homogeneous[:,:2]/homogeneous[:,2:]-p['k2'][j],axis=1)
        return np.where(np.isfinite(e),e,np.inf)

    def timed_call(fn):
        t=time.perf_counter_ns();fn();return (time.perf_counter_ns()-t)/1e6

    opt_datasets={'synthetic_validation':[opt_pair(p) for p in validation_pairs]}
    if real_pairs:opt_datasets['HPatches_test']=[opt_pair(p) for p in real_pairs]
    opt_accuracy=[];opt_times=[];opt_breakdown=[];opt_error_lists={};opt_checks=[]
    with threadpool_limits(limits=NUMBA_THREADS):
        for dataset,pairs in opt_datasets.items():
            sample_ids=set(np.unique(np.linspace(0,len(pairs)-1,min(TIMING_PAIRS_PER_DATASET,len(pairs)),dtype=int)))
            for pi,p in enumerate(pairs):
                z={method:opt_prepare(p,method) for method in OPT_METHODS}
                # Bit-identical direct encoder required on EVERY actual image pair.
                for key in ('pa','pb'):
                    np.testing.assert_array_equal(z['thermo_split'][key],z['thermo_direct'][key])
                matches={method:opt_match(z[method],method) for method in OPT_METHODS}
                for expected,actual in zip(matches['thermo_split'],matches['thermo_direct']):np.testing.assert_array_equal(expected,actual)
                base='blob_mean_split_serial'
                base_allowed=opt_select(z[base],base)
                base_nn=opt_kernel(z[base],base_allowed,base)
                for method in ['blob_mean_direct_serial','blob_mean_direct_parallel']:
                    allowed=opt_select(z[method],method)
                    np.testing.assert_array_equal(base_allowed,allowed)
                    candidate_nn=opt_kernel(z[method],allowed,method)
                    for a,b in zip(base_nn[:2],candidate_nn[:2]):np.testing.assert_array_equal(a,b)
                    for a,b in zip(matches[base],matches[method]):np.testing.assert_array_equal(a,b)
                # BIN-mean changes candidates but parallel vs serial must still agree.
                method='blob_binmean_direct_parallel';allowed=opt_select(z[method],method)
                par=opt_kernel(z[method],allowed,method)
                seq=opt_kernel(z[method],allowed,'blob_binmean_direct_serial')
                for a,b in zip(par[:2],seq[:2]):np.testing.assert_array_equal(a,b)
                opt_checks.append(dict(dataset=dataset,pair=pi,encoding_exact=True,mean_candidates_exact=True,both_nn_exact=True))
                ref=matches['thermo_split'];ref_ok=opt_errors(p,ref)<GEOMETRIC_THRESHOLD
                ref_i,ref_j=ref[0][ref_ok],ref[1][ref_ok]
                n,m=len(p['x']),len(p['y']);total=n*m
                for method in OPT_METHODS:
                    ij=matches[method];err=opt_errors(p,ij)
                    opt_error_lists.setdefault((dataset,method),[]).extend(err.tolist())
                    count=total;kept=len(ref_i)
                    extra=0
                    if method.startswith('blob_'):
                        a,b=z[method]['a'],z[method]['b'];allowed=opt_select(z[method],method)
                        count=int((allowed*np.diff(a['offsets'])[:,None]*np.diff(b['offsets'])[None,:]).sum())
                        kept=int(allowed[a['labels'][ref_i],b['labels'][ref_j]].sum())
                        assert opt_kernel(z[method],allowed,method)[2]==count
                        if method.endswith('parallel'):extra=len(a['occupied'])*m*10 # uint16+int64 local columns
                    opt_accuracy.append(dict(dataset=dataset,pair=pi,name=p['name'],method=method,n=n,m=m,
                        mutual=len(ij[0]),correct=int((err<GEOMETRIC_THRESHOLD).sum()),
                        compared=count,total_pairs=total,reference_correct=len(ref_i),reference_kept=kept,
                        parallel_column_scratch_bytes=extra))
                if pi in sample_ids and total:
                    stage_a,stage_b=layouts(p)
                    qa,qb=quantize_array(p['x'],OPT_EDGES),quantize_array(p['y'],OPT_EDGES)
                    stages={
                        'bin_quantize_ms':lambda:(quantize_array(p['x'],OPT_EDGES),quantize_array(p['y'],OPT_EDGES)),
                        'pack_existing_bins_ms':lambda:(pack_thermometer64(qa),pack_thermometer64(qb)),
                        'fused_encode_pack_ms':lambda:(direct_thermo(p['x'],OPT_EDGES),direct_thermo(p['y'],OPT_EDGES)),
                        'grid_layout_ms':lambda:layouts(p),
                        'float_blob_mean_ms':lambda:(original_float_mean(p['x'],stage_a),original_float_mean(p['y'],stage_b)),
                        'fused_encode_pack_binmean_ms':lambda:(direct_thermo_binmean(p['x'],OPT_EDGES,stage_a['order'],stage_a['offsets']),direct_thermo_binmean(p['y'],OPT_EDGES,stage_b['order'],stage_b['offsets']))}
                    work={}
                    for method in OPT_METHODS:
                        got=opt_total(p,method)
                        for a,b in zip(matches[method],got):np.testing.assert_array_equal(a,b)
                        work[method]={'cached_match_ms':lambda method=method:opt_match(z[method],method),
                                      'prepare_ms':lambda method=method:opt_prepare(p,method),
                                      'total_ms':lambda method=method:opt_total(p,method)}
                        if method.startswith('blob_'):
                            allowed=opt_select(z[method],method)
                            stages[method+'_candidate_search_ms']=lambda method=method:opt_select(z[method],method)
                            stages[method+'_nn_kernel_ms']=lambda method=method,allowed=allowed:opt_kernel(z[method],allowed,method)
                    for _ in range(WARMUP):
                        for group in work.values():
                            for fn in group.values():fn()
                        for fn in stages.values():fn()
                    trng=np.random.default_rng(SEED+pi)
                    for repeat in range(REPEATS):
                        for method in trng.permutation(OPT_METHODS):
                            values={phase:timed_call(work[method][phase]) for phase in trng.permutation(list(work[method]))}
                            if method=='float_cosine':values['prepare_ms']=0.0
                            opt_times.append(dict(dataset=dataset,pair=pi,method=str(method),repeat=repeat,**values))
                        for stage in trng.permutation(list(stages)):
                            opt_breakdown.append(dict(dataset=dataset,pair=pi,stage=str(stage),repeat=repeat,ms=timed_call(stages[stage])))
                print('Verified / evaluated:',dataset,pi+1,'/',len(pairs))
        opt_threadpools=threadpool_info()
    opt_accuracy_df=pd.DataFrame(opt_accuracy);opt_timing_raw=pd.DataFrame(opt_times);opt_stages_raw=pd.DataFrame(opt_breakdown)
    rows=[]
    for (dataset,method),df in opt_accuracy_df.groupby(['dataset','method'],sort=False):
        mutual=int(df.mutual.sum());correct=int(df.correct.sum());total=int(df.total_pairs.sum());ref=int(df.reference_correct.sum())
        err=opt_error_lists[dataset,method]
        rows.append(dict(dataset=dataset,method=method,mutual=mutual,correct=correct,
            precision_pct=100*correct/mutual if mutual else np.nan,median_px=np.median(err) if err else np.nan,
            candidate_pct=100*df.compared.sum()/total if total else np.nan,
            reference_keep_pct=100*df.reference_kept.sum()/ref if ref else np.nan))
    opt_summary=pd.DataFrame(rows)
    if len(opt_timing_raw):
        opt_per_pair=opt_timing_raw.groupby(['dataset','pair','method'])[['cached_match_ms','prepare_ms','total_ms']].median().reset_index()
        opt_summary=opt_summary.merge(opt_per_pair.groupby(['dataset','method'])[['cached_match_ms','prepare_ms','total_ms']].median().reset_index(),on=['dataset','method'])
        for dataset,df in opt_summary.groupby('dataset'):
            ref=df.set_index('method');mask=opt_summary.dataset==dataset
            opt_summary.loc[mask,'total_speedup_vs_float']=ref.loc['float_cosine','total_ms']/opt_summary.loc[mask,'total_ms']
            opt_summary.loc[mask,'total_speedup_vs_thermo_direct']=ref.loc['thermo_direct','total_ms']/opt_summary.loc[mask,'total_ms']
            opt_summary.loc[mask,'cached_speedup_vs_thermo_direct']=ref.loc['thermo_direct','cached_match_ms']/opt_summary.loc[mask,'cached_match_ms']
        opt_stages=opt_stages_raw.groupby(['dataset','pair','stage']).ms.median().groupby(['dataset','stage']).median().reset_index()
    else:
        opt_per_pair=pd.DataFrame();opt_stages=pd.DataFrame()
        for col in ('cached_match_ms','prepare_ms','total_ms'):opt_summary[col]=np.nan
    display(opt_summary.round(4));display(opt_stages.round(4))
    print('PASS on all evaluation pairs: old/new encoding bit equality and mean Blob candidates + BOTH NN arrays + MNN equality.')
    print('BIN-mean is a separate approximate candidate representation; check candidate recall and correct count.')
    print('Speedup >1 means faster in THIS run. Total is directly timed; stage medians are diagnostic and need not sum to total.')
    print('Reference keep is candidate survival, not final-match recall; full-search methods have 100% candidate survival by definition.')
    print('All CPU thread limits:',NUMBA_THREADS,'; serial Blob baseline is one thread. XFeat/JIT/verification/I/O excluded.')
    opt_fig,opt_axes=plt.subplots(len(opt_datasets),2,figsize=(16,5*len(opt_datasets)),squeeze=False)
    for row,dataset in enumerate(opt_datasets):
        d=opt_summary[opt_summary.dataset==dataset].set_index('method').reindex(OPT_METHODS)
        d[['cached_match_ms','total_ms']].plot.barh(ax=opt_axes[row,0],title=dataset+' (ms)')
        d[['precision_pct','candidate_pct']].plot.barh(ax=opt_axes[row,1],title='Precision / comparisons (%)')
        opt_axes[row,1].set_xlim(0,105)
    opt_fig.tight_layout();display(opt_fig);plt.close(opt_fig)
else:
    print("Blob benchmark skipped. Continue to exact Thermometer sections 12–13.")


Blob benchmark skipped. Continue to exact Thermometer sections 12–13.


## 11. 최적화 결과 저장
서울시각의 새 optimized_blob 폴더에 표·그래프·반복 시간·경계·설정을 저장합니다.

In [36]:
if RUN_BLOB_BENCHMARKS:
    from datetime import datetime
    from zoneinfo import ZoneInfo
    import numba
    run_id=datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S_%f')
    optimized_result_dir=data_dir/'experiment_results'/('optimized_blob_'+run_id)
    optimized_result_dir.mkdir(parents=True,exist_ok=False)
    for filename,df in [('summary.csv',opt_summary),('accuracy_per_pair.csv',opt_accuracy_df),
                        ('timing_raw.csv',opt_timing_raw),('timing_per_pair.csv',opt_per_pair),
                        ('stage_times.csv',opt_stages),('stage_times_raw.csv',opt_stages_raw),
                        ('exact_checks.csv',pd.DataFrame(opt_checks))]:
        df.to_csv(optimized_result_dir/filename,index=False)
    np.save(optimized_result_dir/'fixed_quantile_edges.npy',OPT_EDGES)
    opt_fig.savefig(optimized_result_dir/'comparison.png',dpi=160,bbox_inches='tight')
    def opt_records(df):return json.loads(df.to_json(orient='records'))
    report={'experiment':'exact encoding and mean Blob acceleration plus experimental BIN mean',
        'run_id_seoul':run_id,'seed':SEED,'top_k':TOP_K,'levels':8,'dimensions':64,
        'grid':OPT_GRID,'top_blobs':OPT_TOP_K,'expand':False,
        'cpu_threads':NUMBA_THREADS,'serial_blob_threads':1,
        'warmup':WARMUP,'repeats':REPEATS,'timing_pairs_per_dataset':TIMING_PAIRS_PER_DATASET,
        'threshold_resized_px':GEOMETRIC_THRESHOLD,'platform':platform.platform(),
        'versions':{'numpy':np.__version__,'torch':torch.__version__,'numba':numba.__version__},
        'threadpools':opt_threadpools,'extraction_device':device,
        'edges_sha256':hashlib.sha256(OPT_EDGES.tobytes()).hexdigest(),
        'source_names':[name for name,_ in image_items],
        'calibration_sources':[name for i,(name,_) in enumerate(image_items) if i not in validation_ids],
        'validation_sources':[name for i,(name,_) in enumerate(image_items) if i in validation_ids],
        'summary':opt_records(opt_summary),'exact_checks':opt_checks,
        'stage_times':opt_records(opt_stages),
        'notes':['All exact-change assertions passed on all evaluation pairs.',
                 'BIN mean can change candidate graph; it is not an exact optimization.',
                 'Same-run float and full Thermometer baselines; extraction/JIT excluded.',
                 'Grid 6/top4 fixed from prior synthetic validation; no new test-set tuning.',
                 'Descriptor payload: float256B; thermometer56B. Extra Blob means and indices are additional.']}
    with open(optimized_result_dir/'results.json','w') as f:json.dump(report,f,indent=2,allow_nan=False)
    print('OPTIMIZED RESULT FOLDER:',optimized_result_dir)
    for path in sorted(optimized_result_dir.iterdir()):print(path.name)
else:
    print("Blob benchmark skipped. Continue to exact Thermometer sections 12–13.")


Blob benchmark skipped. Continue to exact Thermometer sections 12–13.


## 12. 정확한 양방향 동시 갱신과 블록 커널
작은 입력·빈 입력·동점·블록 경계를 검사하고 새 커널의 POPCOUNT 기계어도 출력합니다.

In [37]:
# Every pair evaluated once; ties choose the lowest original index.
@njit(inline='always')
def exact_distance7(a,b,i,j):
    d=0
    for k in range(7):
        d+=np.int64(thermometer_popcount64(a[i,k]^b[j,k]))
    return d

@njit(cache=False)
def thermo_fused_nn(a,b):
    n,m=len(a),len(b)
    r=np.full(n,-1,np.int64);c=np.full(m,-1,np.int64)
    rd=np.full(n,449,np.uint16);cd=np.full(m,449,np.uint16)
    for i in range(n):
        best=449;idx=-1
        for j in range(m):
            d=exact_distance7(a,b,i,j)
            if d<best:best=d;idx=j
            if d<cd[j]:cd[j]=d;c[j]=i
        r[i]=idx;rd[i]=best
    return r,c

@njit(cache=False)
def thermo_blocked_nn(a,b,block):
    n,m=len(a),len(b)
    r=np.full(n,-1,np.int64);c=np.full(m,-1,np.int64)
    rd=np.full(n,449,np.uint16);cd=np.full(m,449,np.uint16)
    for i0 in range(0,n,block):
        for j0 in range(0,m,block):
            for i in range(i0,min(i0+block,n)):
                best=np.int64(rd[i]);idx=r[i]
                for j in range(j0,min(j0+block,m)):
                    d=exact_distance7(a,b,i,j)
                    if d<best or (d==best and (idx<0 or j<idx)):best=d;idx=j
                    if d<cd[j] or (d==cd[j] and (c[j]<0 or i<c[j])):cd[j]=d;c[j]=i
                rd[i]=best;r[i]=idx
    return r,c

@njit(parallel=True,cache=False)
def thermo_blocked_parallel_nn(a,b,block):
    n,m=len(a),len(b);ng=(n+block-1)//block
    r=np.full(n,-1,np.int64);rd=np.full(n,449,np.uint16)
    local_d=np.full((ng,m),449,np.uint16)
    local_i=np.full((ng,m),-1,np.int64)
    # Each A stripe owns disjoint r rows and a private B winner array.
    for g in prange(ng):
        i0=g*block
        for j0 in range(0,m,block):
            for i in range(i0,min(i0+block,n)):
                best=np.int64(rd[i]);idx=r[i]
                for j in range(j0,min(j0+block,m)):
                    d=exact_distance7(a,b,i,j)
                    if d<best or (d==best and (idx<0 or j<idx)):best=d;idx=j
                    if d<local_d[g,j] or (d==local_d[g,j] and (local_i[g,j]<0 or i<local_i[g,j])):
                        local_d[g,j]=d;local_i[g,j]=i
                rd[i]=best;r[i]=idx
    c=np.full(m,-1,np.int64)
    for j in prange(m):
        best=449;idx=-1
        for g in range(ng):
            d=local_d[g,j];i=local_i[g,j]
            if i>=0 and (d<best or (d==best and (idx<0 or i<idx))):best=d;idx=i
        c[j]=idx
    return r,c

def exact_mnn(r,c):
    i=np.flatnonzero(r>=0)
    i=i[c[r[i]]==i]
    return i,r[i]

def dense_nn_oracle(a,b):
    n,m=len(a),len(b)
    if not n or not m:return np.full(n,-1,np.int64),np.full(m,-1,np.int64)
    d=pairwise_thermometer64(a,b)
    return d.argmin(1),d.argmin(0)

def check_exact_kernels():
    rng=np.random.default_rng(1707)
    for n,m in [(0,0),(0,5),(7,0),(1,1),(31,65),(65,33),(129,137)]:
        for ties in (False,True):
            a=pack_thermometer64(rng.integers(0,8,(n,64),dtype=np.uint8))
            b=pack_thermometer64(rng.integers(0,8,(m,64),dtype=np.uint8))
            if ties:a[:]=0;b[:]=0
            ref=dense_nn_oracle(a,b)
            candidates=[thermo_fused_nn(a,b)]
            for block in (1,32,64,128):
                candidates.extend([thermo_blocked_nn(a,b,block),thermo_blocked_parallel_nn(a,b,block)])
            for got in candidates:
                for x,y in zip(ref,got):np.testing.assert_array_equal(x,y)
                for x,y in zip(exact_mnn(*ref),exact_mnn(*got)):np.testing.assert_array_equal(x,y)
    # Includes maximum distance 448, high bit, and winners tied across tiles.
    a=pack_thermometer64(np.zeros((70,64),np.uint8))
    b=pack_thermometer64(np.full((67,64),7,np.uint8))
    for fn,args in [(thermo_fused_nn,()),(thermo_blocked_nn,(32,)),(thermo_blocked_parallel_nn,(32,))]:
        r,c=fn(a,b,*args);assert np.all(r==0) and np.all(c==0)
    print('PASS: exact BOTH NN + MNN, empty inputs, non-square/tile tails, ties, maximum distance 448.')

check_exact_kernels()
for _name,_fn in [('fused',thermo_fused_nn),('blocked_serial',thermo_blocked_nn),('blocked_parallel',thermo_blocked_parallel_nn)]:
    POPCOUNT_REPORTS.append(popcount_report(_name,_fn))


PASS: exact BOTH NN + MNN, empty inputs, non-square/tile tails, ties, maximum distance 448.
fused : YES_NATIVE_POPCOUNT | LLVM ctpop: True
    popcntq	%rcx, %rcx
    popcntq	%rax, %rax
    popcntq	%rcx, %rcx
    popcntq	%rbx, %rax
blocked_serial : YES_NATIVE_POPCOUNT | LLVM ctpop: True
    popcntq	%rdx, %rdx
    popcntq	%rsi, %rsi
    popcntq	%rcx, %rcx
    popcntq	%rdx, %rdx
blocked_parallel : YES_NATIVE_POPCOUNT | LLVM ctpop: True
    popcntq	%rax, %rax
    popcntq	%rcx, %rcx
    popcntq	%rax, %rax
    popcntq	%rcx, %rcx


## 13. 정확도·속도 비교 및 저장
모든 실제 쌍의 결과 일치 검증. 블록 크기와 스레드별 시간, 5 ms 목표 여부를 출력합니다.

In [38]:
import time,json,platform,hashlib
import pandas as pd
from numba import set_num_threads,get_num_threads
from threadpoolctl import threadpool_limits,threadpool_info
from datetime import datetime
from zoneinfo import ZoneInfo

# Default experiment: no Blob, no retraining, no approximate pruning.
EXACT_BLOCKS=(32,64,128)
EXACT_THREADS=tuple(sorted(set([1,min(2,numba.config.NUMBA_NUM_THREADS)])))
EXACT_DEFAULT_THREADS=min(NUMBA_THREADS,numba.config.NUMBA_NUM_THREADS)
EXACT_TARGET_MS=5.0

def exact_array(x):
    return np.ascontiguousarray(x.detach().cpu().numpy() if hasattr(x,'detach') else x,dtype=np.float32)
EXACT_EDGES=exact_array(base_edges_cpu)

def exact_pair(p):
    return dict(x=exact_array(p['d1']),y=exact_array(p['d2']),k1=exact_array(p['k1']),
                k2=exact_array(p['k2']),H=np.asarray(p['H']),name=p.get('name',''))

def exact_prepare(p):return direct_thermo(p['x'],EXACT_EDGES),direct_thermo(p['y'],EXACT_EDGES)

def exact_error(p,ij):
    i,j=ij
    h=np.c_[p['k1'][i],np.ones(len(i))]@p['H'].T
    with np.errstate(divide='ignore',invalid='ignore'):
        e=np.linalg.norm(h[:,:2]/h[:,2:]-p['k2'][j],axis=1)
    return np.where(np.isfinite(e),e,np.inf)

def exact_ms(fn):
    start=time.perf_counter_ns();fn();return (time.perf_counter_ns()-start)/1e6

# Same thread setting for float/dense baselines and encoder. Serial NN uses one thread.
variants={'float_cosine':('float',0,EXACT_DEFAULT_THREADS),
          'thermo_dense_existing':('dense',0,EXACT_DEFAULT_THREADS),
          'thermo_fused_serial':('fused',0,EXACT_DEFAULT_THREADS)}
for b in EXACT_BLOCKS:variants[f'thermo_block{b}_serial']=('blocked',b,EXACT_DEFAULT_THREADS)
for t in EXACT_THREADS:variants[f'thermo_block64_parallel_t{t}']=('parallel',64,t)

def exact_nn(pa,pb,spec):
    kind,block,threads=spec
    if kind=='dense':return dense_nn_oracle(pa,pb)
    if kind=='fused':return thermo_fused_nn(pa,pb)
    if kind=='blocked':return thermo_blocked_nn(pa,pb,block)
    return thermo_blocked_parallel_nn(pa,pb,block)

def exact_match(p,encoded,spec):
    kind,block,threads=spec
    if kind=='float':return mutual_from_scores(p['x']@p['y'].T,True)
    # Preserve the exact existing baseline path (matrix + NumPy MNN).
    if kind=='dense':return mutual_from_scores(pairwise_thermometer64(*encoded),False)
    return exact_mnn(*exact_nn(*encoded,spec))

def exact_total(p,spec):
    return exact_match(p,None if spec[0]=='float' else exact_prepare(p),spec)

exact_datasets={'synthetic_validation':[exact_pair(p) for p in validation_pairs]}
if real_pairs:exact_datasets['HPatches_test']=[exact_pair(p) for p in real_pairs]
accuracy=[];timings=[];checks=[];errors={}
_saved_threads=get_num_threads()
try:
  with threadpool_limits(limits=EXACT_DEFAULT_THREADS):
    for dataset,pairs in exact_datasets.items():
      sample_ids=set(np.unique(np.linspace(0,len(pairs)-1,min(TIMING_PAIRS_PER_DATASET,len(pairs)),dtype=int))) if pairs else set()
      for pi,p in enumerate(pairs):
        set_num_threads(EXACT_DEFAULT_THREADS)
        pa,pb=encoded=exact_prepare(p)
        for x,packed in zip((p['x'],p['y']),encoded):
            q=np.ascontiguousarray((x[:,:,None]>EXACT_EDGES[None,:,:]).sum(2),dtype=np.uint8)
            np.testing.assert_array_equal(packed,pack_thermometer64(q))
        ref_nn=dense_nn_oracle(pa,pb);ref_mnn=exact_mnn(*ref_nn)
        expected={}
        for name,spec in variants.items():
            set_num_threads(spec[2])
            if spec[0]!='float':
                for a,b in zip(ref_nn,exact_nn(pa,pb,spec)):np.testing.assert_array_equal(a,b)
            ij=exact_match(p,encoded,spec);expected[name]=ij
            if spec[0]!='float':
                for a,b in zip(ref_mnn,ij):np.testing.assert_array_equal(a,b)
                checks.append(dict(dataset=dataset,pair=pi,method=name,bits_exact=True,both_nn_exact=True,mnn_exact=True))
            e=exact_error(p,ij);errors.setdefault((dataset,name),[]).extend(e.tolist())
            accuracy.append(dict(dataset=dataset,pair=pi,name=p['name'],method=name,mutual=len(e),
                correct=int((e<GEOMETRIC_THRESHOLD).sum()),n=len(pa),m=len(pb),
                distance_matrix_bytes=2*len(pa)*len(pb) if spec[0]=='dense' else (4*len(pa)*len(pb) if spec[0]=='float' else 0),
                private_column_bytes=((len(pa)+spec[1]-1)//spec[1])*len(pb)*10 if spec[0]=='parallel' else 0))
        if pi in sample_ids and len(pa) and len(pb):
            # Precompile/warm all signatures, then randomize variant and phase order.
            for name,spec in variants.items():
                set_num_threads(spec[2])
                for _ in range(WARMUP):exact_total(p,spec);exact_match(p,encoded,spec)
                for a,b in zip(expected[name],exact_total(p,spec)):np.testing.assert_array_equal(a,b)
            rng=np.random.default_rng(SEED+pi)
            for repeat in range(REPEATS):
                for name in rng.permutation(list(variants)):
                    spec=variants[name];set_num_threads(spec[2]) # outside timed region
                    work={'prepare_ms':lambda:exact_prepare(p),
                          'cached_match_ms':lambda:exact_match(p,encoded,spec),
                          'total_ms':lambda:exact_total(p,spec)}
                    values={phase:exact_ms(work[phase]) for phase in rng.permutation(list(work))}
                    if spec[0]=='float':values['prepare_ms']=0.0
                    timings.append(dict(dataset=dataset,pair=pi,method=str(name),repeat=repeat,
                        numba_threads=spec[2],nn_threads=1 if spec[0] in ('fused','blocked') else spec[2],**values))
        print('Exact NN/MNN verified:',dataset,pi+1,'/',len(pairs))
finally:set_num_threads(_saved_threads)

exact_accuracy=pd.DataFrame(accuracy);exact_raw=pd.DataFrame(timings);exact_checks=pd.DataFrame(checks)
rows=[]
for (dataset,method),g in exact_accuracy.groupby(['dataset','method'],sort=False):
    mutual=int(g.mutual.sum());correct=int(g.correct.sum());e=errors[dataset,method]
    rows.append(dict(dataset=dataset,method=method,mutual=mutual,correct=correct,
        precision_pct=100*correct/mutual if mutual else np.nan,median_px=np.median(e) if e else np.nan,
        exact_vs_thermo=method!='float_cosine'))
exact_summary=pd.DataFrame(rows)
exact_per_pair=pd.DataFrame()
if len(exact_raw):
    phases=['prepare_ms','cached_match_ms','total_ms']
    exact_per_pair=exact_raw.groupby(['dataset','pair','method'])[phases].median().reset_index()
    summary=exact_per_pair.groupby(['dataset','method'])[phases].median().reset_index()
    exact_summary=exact_summary.merge(summary,on=['dataset','method'])
    for dataset,g in exact_summary.groupby('dataset'):
        base=g.set_index('method');mask=exact_summary.dataset==dataset
        exact_summary.loc[mask,'speedup_vs_dense']=base.loc['thermo_dense_existing','total_ms']/exact_summary.loc[mask,'total_ms']
        exact_summary.loc[mask,'speedup_vs_float']=base.loc['float_cosine','total_ms']/exact_summary.loc[mask,'total_ms']
    exact_summary['under_5ms']=exact_summary.total_ms<=EXACT_TARGET_MS
else:
    for col in ['prepare_ms','cached_match_ms','total_ms']:exact_summary[col]=np.nan
print('POPCOUNT diagnostic (NOT_CONFIRMED means inspect assembly):')
display(pd.DataFrame(POPCOUNT_REPORTS))
display(exact_summary.round(4))
print('PASS: all Thermometer variants preserve BOTH nearest-neighbor arrays and MNN on ALL evaluation pairs.')
print('under_5ms is based on dataset median total time, not a per-pair guarantee. Check timing_per_pair.csv.')
print('Timing excludes XFeat extraction, JIT, checks, thread-setting calls and I/O; includes allocations and MNN extraction.')
print('Serial NN still uses configured parallel encoding. Parallel t1/t2 also changes encoder threads; see prepare_ms.')
print('All block/thread variants are reported; HPatches is not used to select a winner for deployment.')

run_id=datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S_%f')
exact_result_dir=data_dir/'experiment_results'/('thermo_exact_'+run_id)
exact_result_dir.mkdir(parents=True,exist_ok=False)
for name,df in [('summary.csv',exact_summary),('accuracy_per_pair.csv',exact_accuracy),('timing_raw.csv',exact_raw),
                ('timing_per_pair.csv',exact_per_pair),('exact_checks.csv',exact_checks),('popcount_report.csv',pd.DataFrame(POPCOUNT_REPORTS))]:
    df.to_csv(exact_result_dir/name,index=False)
for name,asm in POPCOUNT_ASSEMBLY.items():(exact_result_dir/(name+'.asm')).write_text(asm)
np.save(exact_result_dir/'fixed_quantile_edges.npy',EXACT_EDGES)
report=dict(experiment='exact Thermometer fused and tiled NN',variants=variants,target_ms=EXACT_TARGET_MS,
    cpu=str(llvm_binding.get_host_cpu_name()),platform=platform.platform(),numba=numba.__version__,numpy=np.__version__,
    repeats=REPEATS,warmup=WARMUP,timing_pairs_per_dataset=TIMING_PAIRS_PER_DATASET,
    geometric_threshold=GEOMETRIC_THRESHOLD,seed=SEED,top_k=TOP_K,popcount=POPCOUNT_REPORTS,
    edges_sha256=hashlib.sha256(EXACT_EDGES.tobytes()).hexdigest(),
    summary=json.loads(exact_summary.to_json(orient='records')))
(exact_result_dir/'results.json').write_text(json.dumps(report,indent=2,allow_nan=False))
print('RESULT FOLDER:',exact_result_dir)


Exact NN/MNN verified: synthetic_validation 1 / 18
Exact NN/MNN verified: synthetic_validation 2 / 18
Exact NN/MNN verified: synthetic_validation 3 / 18
Exact NN/MNN verified: synthetic_validation 4 / 18
Exact NN/MNN verified: synthetic_validation 5 / 18
Exact NN/MNN verified: synthetic_validation 6 / 18
Exact NN/MNN verified: synthetic_validation 7 / 18
Exact NN/MNN verified: synthetic_validation 8 / 18
Exact NN/MNN verified: synthetic_validation 9 / 18
Exact NN/MNN verified: synthetic_validation 10 / 18
Exact NN/MNN verified: synthetic_validation 11 / 18
Exact NN/MNN verified: synthetic_validation 12 / 18
Exact NN/MNN verified: synthetic_validation 13 / 18
Exact NN/MNN verified: synthetic_validation 14 / 18
Exact NN/MNN verified: synthetic_validation 15 / 18
Exact NN/MNN verified: synthetic_validation 16 / 18
Exact NN/MNN verified: synthetic_validation 17 / 18
Exact NN/MNN verified: synthetic_validation 18 / 18
Exact NN/MNN verified: HPatches_test 1 / 24
Exact NN/MNN verified: HPatch

,kernel,status,llvm_ctpop,native_instruction_lines,scalar_popcnt_lines,vector_popcnt_lines,examples
0,existing_pairwise_thermometer64,YES_NATIVE_POPCOUNT,True,7,7,0,"[popcntq\t%r13, %rbx, popcntq\t%rax, %rax, pop..."
1,fused,YES_NATIVE_POPCOUNT,True,7,7,0,"[popcntq\t%rcx, %rcx, popcntq\t%rax, %rax, pop..."
2,blocked_serial,YES_NATIVE_POPCOUNT,True,7,7,0,"[popcntq\t%rdx, %rdx, popcntq\t%rsi, %rsi, pop..."
3,blocked_parallel,YES_NATIVE_POPCOUNT,True,7,7,0,"[popcntq\t%rax, %rax, popcntq\t%rcx, %rcx, pop..."


,dataset,method,mutual,correct,precision_pct,median_px,exact_vs_thermo,prepare_ms,cached_match_ms,total_ms,speedup_vs_dense,speedup_vs_float,under_5ms
0,synthetic_validation,float_cosine,8178,5598,68.4519,1.3656,False,0.0000,3.4407,3.5074,2.4413,1.0000,True
1,synthetic_validation,thermo_dense_existing,8069,5404,66.9724,1.4109,True,2.0739,6.5138,8.5624,1.0000,0.4096,False
2,synthetic_validation,thermo_fused_serial,8069,5404,66.9724,1.4109,True,2.0437,4.3874,6.4492,1.3277,0.5439,False
3,synthetic_validation,thermo_block32_serial,8069,5404,66.9724,1.4109,True,2.0781,6.9806,8.9826,0.9532,0.3905,False
4,synthetic_validation,thermo_block64_serial,8069,5404,66.9724,1.4109,True,2.0467,6.6887,8.7873,0.9744,0.3991,False
5,synthetic_validation,thermo_block128_serial,8069,5404,66.9724,1.4109,True,2.0387,6.8182,8.7826,0.9749,0.3994,False
6,synthetic_validation,thermo_block64_parallel_t1,8069,5404,66.9724,1.4109,True,2.0142,7.4824,9.6027,0.8917,0.3652,False
7,synthetic_validation,thermo_block64_parallel_t2,8069,5404,66.9724,1.4109,True,2.0562,7.3542,9.4081,0.9101,0.3728,False
8,HPatches_test,float_cosine,10607,7138,67.2952,1.1764,False,0.0000,10.6020,10.6495,1.2029,1.0000,False
9,HPatches_test,thermo_dense_existing,10510,6924,65.8801,1.2508,True,3.6274,8.9549,12.8104,1.0000,0.8313,False


PASS: all Thermometer variants preserve BOTH nearest-neighbor arrays and MNN on ALL evaluation pairs.
under_5ms is based on dataset median total time, not a per-pair guarantee. Check timing_per_pair.csv.
Timing excludes XFeat extraction, JIT, checks, thread-setting calls and I/O; includes allocations and MNN extraction.
Serial NN still uses configured parallel encoding. Parallel t1/t2 also changes encoder threads; see prepare_ms.
All block/thread variants are reported; HPatches is not used to select a winner for deployment.
RESULT FOLDER: /content/drive/MyDrive/Colab Notebooks/CSE527_datasets/experiment_results/thermo_exact_20261005_203628_471955
